#Data ingestions for rainfall, temperature and orders volumes

##order ingested

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.bronze_orders AS
SELECT *
FROM read_files(
    '/Volumes/dp_750/default/fulfulment-pulse/fictitious_orders_melbourne.csv',
    format => 'csv',
    header => true,
    Schema => 'order_id STRING,
    order_timestamp TIMESTAMP,
    delivery_zone STRING,
    item_count INT,
    promised_window STRING,
    fulfillment_stage STRING,
    basket_category STRING,
    simulated_delay_flag BOOLEAN,
    is_synthetic BOOLEAN
    '
);

In [0]:
%sql
SELECT * 
FROM dp_750.default.bronze_orders

##temperature ingested

In [0]:
%sql

CREATE OR REPLACE TABLE dp_750.default.bronze_temperature AS 
SELECT *
FROM read_files(
    '/Volumes/dp_750/default/fulfulment-pulse/Temperature_Data.csv',
    format => 'csv',
    header => true,
    schema => 'Product_code STRING,
    Bureau_of_Meteorology_station_number STRING,
    Year INT,
    Month INT,
    Day INT,
    Maximum_temperature_Degree_C DECIMAL,
    Days_of_accumulation_of_maximum_temperature DECIMAL,
    Quality STRING'
);

In [0]:
%sql
SELECT * 
FROM dp_750.default.bronze_temperature

##rainfall ingested

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.bronze_rainfall AS
SELECT *
FROM read_files(
    '/Volumes/dp_750/default/fulfulment-pulse/rainfall_data.csv',   
    format => 'csv',
    header => true,
    schema => 'Product_code STRING,
    Bureau_of_Meteorology_station_number STRING,
    Year INT,
    Month INT,
    Day INT,
    Rainfall_amount_millimetres DECIMAL,
    Days_of_accumulation_of_rainfall DECIMAL,
    Quality STRING');

 

In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_rainfall

#Data cleaning orders

In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_orders
WHERE (order_id, order_timestamp, delivery_zone, item_count) IS NULL

In [0]:
%sql
SELECT order_id
FROM dp_750.default.bronze_orders
WHERE (item_count = 0) OR (item_count REGEXP '.');


In [0]:
%sql
SELECT order_id
FROM dp_750.default.bronze_orders
WHERE (item_count % 1 <> 0) OR (item_count <= 0)

In [0]:
%sql
SELECT order_id
FROM dp_750.default.bronze_orders
GROUP BY order_id
HAVING count(order_id) > 1


In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_orders 
WHERE is_synthetic = FALSE OR is_synthetic IS NULL

In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_orders 
WHERE (delivery_zone IS NULL) OR TRIM(delivery_zone) = ""

In [0]:
%sql
SELECT order_timestamp
FROM dp_750.default.bronze_orders
WHERE order_timestamp < '2026-01-01T08:10:00.000+00:00'

In [0]:
%sql
SELECT order_timestamp
FROM dp_750.default.bronze_orders
WHERE order_timestamp > '2026-01-01T08:10:00.000+00:00'

## bronze orders -> silver orders table 

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.silver_orders AS
SELECT *
FROM read_files(
    '/Volumes/dp_750/default/fulfulment-pulse/fictitious_orders_melbourne.csv',
    format => 'csv',
    header => true,
    Schema => 'order_id STRING,
    order_timestamp TIMESTAMP,
    delivery_zone STRING,
    item_count INT,
    promised_window STRING,
    fulfillment_stage STRING,
    basket_category STRING,
    simulated_delay_flag BOOLEAN,
    is_synthetic BOOLEAN
    '
);

## Data cleaning temperature

In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_temperature
WHERE Maximum_temperature_Degree_C IS NULL


In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_temperature

In [0]:
%sql
SELECT COUNT(Product_code), COUNT(Bureau_of_Meteorology_station_number), COUNT(Maximum_temperature_Degree_C), COUNT(Days_of_accumulation_of_maximum_temperature), 
COUNT(Quality),
COUNT(YEAR), 
COUNT(MONTH), 
COUNT(DAY)
FROM dp_750.default.bronze_temperature
GROUP BY Product_code, Bureau_of_Meteorology_station_number, Maximum_temperature_Degree_C, Days_of_accumulation_of_maximum_temperature
, Quality, YEAR, MONTH, DAY

HAVING COUNT(*) > 1
 

In [0]:
%sql
SELECT COUNT(Product_code)
FROM dp_750.default.bronze_temperature
GROUP BY Product_code


In [0]:
%sql
SELECT COUNT(Bureau_of_Meteorology_station_number)
FROM dp_750.default.bronze_temperature
GROUP BY Bureau_of_Meteorology_station_number


In [0]:
%sql
SELECT COUNT(DISTINCT(YEAR, MONTH, DAY))
FROM dp_750.default.bronze_temperature
GROUP BY YEAR, MONTH, DAY
HAVING COUNT(DISTINCT(YEAR, MONTH, DAY)) > 1

In [0]:
%sql
SELECT 
COUNT(*) - COUNT(product_code) AS NULLS_IN_PRODUCT_CODE,
COUNT(*) - COUNT(Bureau_of_Meteorology_station_number) AS NULLS_IN_Bureau_of_Meteorology_station_number, 
COUNT(*) - COUNT(Year) AS NULLS_IN_year,
COUNT(*) - COUNT(Month) AS NULLS_IN_month,
COUNT(*) - COUNT(day) AS NULLS_IN_day,
COUNT(*) - COUNT(Maximum_temperature_Degree_C) AS Maximum_temperature_Degree_C,
COUNT(*) - COUNT(Days_of_accumulation_of_maximum_temperature) AS NULLS_IN_Days_of_accumulation_of_maximum_temperature, 
COUNT(*) - COUNT(Quality) AS NULLS_IN_Q

FROM dp_750.default.bronze_temperature




In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_temperature
GROUP BY product_code,
Bureau_of_Meteorology_station_number,
Year,
Month,
day,
Maximum_temperature_Degree_C,
Days_of_accumulation_of_maximum_temperature,
Quality
HAVING COUNT(*) > 1



In [0]:
%sql
SELECT 
COUNT(*) - COUNT(product_code) AS NULLS_IN_PRODUCT_CODE,
COUNT(*) - COUNT(Bureau_of_Meteorology_station_number) AS NULLS_IN_Bureau_of_Meteorology_station_number, 
COUNT(*) - COUNT(Year) AS NULLS_IN_year,
COUNT(*) - COUNT(Month) AS NULLS_IN_month,
COUNT(*) - COUNT(day) AS NULLS_IN_day,
COUNT(*) - COUNT(Maximum_temperature_Degree_C) AS Maximum_temperature_Degree_C,
COUNT(*) - COUNT(Days_of_accumulation_of_maximum_temperature) AS NULLS_IN_Days_of_accumulation_of_maximum_temperature, 
COUNT(*) - COUNT(Quality) AS NULLS_IN_Q

FROM dp_750.default.bronze_temperature


In [0]:
%sql
SELECT make_date(YEAR, MONTH, DAY) AS observation_date 
FROM dp_750.default.bronze_temperature



In [0]:
%sql
SELECT
    Bureau_of_Meteorology_station_number AS station_id,
    Year,
    Month,
    Day,
    Maximum_temperature_Degree_C AS temperature_c,
    Quality
FROM dp_750.default.bronze_temperature
WHERE Maximum_temperature_Degree_C < -10
   OR Maximum_temperature_Degree_C > 50
ORDER BY temperature_c;

In [0]:
%sql
SELECT
    Year,
    Month,
    Day,
    Maximum_temperature_Degree_C,
    Days_of_accumulation_of_maximum_temperature
FROM dp_750.default.bronze_temperature
WHERE Days_of_accumulation_of_maximum_temperature <= 0
   OR Days_of_accumulation_of_maximum_temperature % 1 <> 0;

In [0]:
%sql
-- 1. Quality flag distribution
SELECT Quality, COUNT(*) AS cnt
FROM dp_750.default.bronze_temperature
GROUP BY Quality
ORDER BY cnt DESC;


In [0]:
%sql
-- 2. Date gap check: find missing days in the time series
WITH date_range AS (
  SELECT make_date(Year, Month, Day) AS obs_date
  FROM dp_750.default.bronze_temperature
),
expected AS (
  SELECT EXPLODE(SEQUENCE(MIN(obs_date), MAX(obs_date), INTERVAL 1 DAY)) AS expected_date
  FROM date_range
)
SELECT e.expected_date AS missing_date
FROM expected e
LEFT JOIN date_range d ON e.expected_date = d.obs_date
WHERE d.obs_date IS NULL
ORDER BY missing_date;


In [0]:
%sql
-- 3. Whitespace check on string columns
SELECT 'Product_code' AS col, COUNT(*) AS affected
FROM dp_750.default.bronze_temperature
WHERE Product_code <> TRIM(Product_code)
UNION ALL
SELECT 'Station_number', COUNT(*)
FROM dp_750.default.bronze_temperature
WHERE Bureau_of_Meteorology_station_number <> TRIM(Bureau_of_Meteorology_station_number)
UNION ALL
SELECT 'Quality', COUNT(*)
FROM dp_750.default.bronze_temperature
WHERE Quality <> TRIM(Quality);


## Data bronze -> silver table for temperature data

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.silver_temperature
AS
SELECT product_code,
Bureau_of_Meteorology_station_number,
make_date (year, month, day) AS observation_date,
Maximum_temperature_Degree_C,
days_of_accumulation_of_maximum_temperature,
quality

FROM dp_750.default.bronze_temperature 





## Data cleaning rainfall

In [0]:
%sql
SELECT * 
FROM dp_750.default.bronze_rainfall

In [0]:
%sql
SELECT COUNT(*) - COUNT(PRODUCT_CODE),
COUNT(*) - COUNT(Bureau_of_Meteorology_station_number
),
COUNT(*) - COUNT(YEAR),
COUNT(*) - COUNT(MONTH),
COUNT(*) - COUNT(DAY),
COUNT(*) - COUNT(Rainfall_amount_millimetres),
COUNT(*) - COUNT(Days_of_accumulation_of_rainfall),
COUNT(*) - COUNT(Quality)
FROM dp_750.default.bronze_rainfall

In [0]:
%sql
SELECT COUNT(*), Product_code, Bureau_of_Meteorology_station_number, Year, month, day
FROM dp_750.default.bronze_rainfall
GROUP BY product_code, Bureau_of_Meteorology_station_number, Year, Month, DAY
HAVING COUNT(Product_code) > 1 
OR COUNT(Bureau_of_Meteorology_station_number) > 1 
OR COUNT(YEAR) > 1
OR COUNT(MONTH) > 1
OR COUNT(DAY) > day

    


In [0]:
%sql
SELECT make_date(YEAR, MONTH, DAY) AS observation_date 
FROM dp_750.default.bronze_rainfall
    

In [0]:
%sql
SELECT Rainfall_amount_millimetres, Days_of_accumulation_of_rainfall
FROM dp_750.default.bronze_rainfall
WHERE Rainfall_amount_millimetres < 0 
OR Days_of_accumulation_of_rainfall < 0 


In [0]:
%sql
SELECT
  CASE
      WHEN Rainfall_amount_millimetres IS NULL THEN TRUE
      ELSE FALSE
  END AS is_rainfall_missing,

  CASE
      WHEN Days_of_accumulation_of_rainfall IS NULL THEN TRUE
      ELSE FALSE
  END AS is_measurement_period_missing,

  CASE
      WHEN Quality IS NULL THEN TRUE
      ELSE FALSE
  END AS is_quality_missing
FROM dp_750.default.bronze_rainfall

## Bronze rainfall -> Silver rainfall table creation

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.silver_rainfall AS
(
  SELECT
    Product_code,
    Bureau_of_Meteorology_station_number,
    make_date(YEAR, MONTH, DAY) AS observation_date,
    Rainfall_amount_millimetres,
1